# Baseline MILP Formulation (Pyomo + GLPK)

Course-project attempt to implement a home healthcare routing model. **The saved GLPK run reports infeasibility**; this notebook is retained to document the initial formulation and its limitations. It should not be described as a successful solution of the original paper.


In [1]:
!apt-get install glpk-utils

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libamd2 libcolamd2 libglpk40 libsuitesparseconfig5
Suggested packages:
  libiodbc2-dev
The following NEW packages will be installed:
  glpk-utils libamd2 libcolamd2 libglpk40 libsuitesparseconfig5
0 upgraded, 5 newly installed, 0 to remove and 18 not upgraded.
Need to get 625 kB of archives.
After this operation, 2,158 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libsuitesparseconfig5 amd64 1:5.10.1+dfsg-4build1 [10.4 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libamd2 amd64 1:5.10.1+dfsg-4build1 [21.6 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libcolamd2 amd64 1:5.10.1+dfsg-4build1 [18.0 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libglpk40 amd64 5.0-1 [361 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy/universe amd64 glpk-ut

In [5]:
from pyomo.environ import *

# Model
model = ConcreteModel()

# Number of patients, nurses, and transportation systems
M = 3  # Number of patients
N = 2  # Number of nurses
K = 1  # Number of transportation systems

# Fixed input data
Dij = [[0, 10, 15], [10, 0, 20], [15, 20, 0]]  # Distances between patients
Ai = [5, 3, 2]  # Drug demand for each patient
Bi = [2, 1, 3]  # Biological samples collected from each patient
Ei = [8, 10, 12]  # Earliest service time for each patient
Li = [20, 18, 22]  # Latest service time for each patient
Wi = [1, 2, 1]  # Service time for each patient
CAPk = [10]  # Transportation system capacity
MDISnk = [[50], [50]]  # Maximum allowable distance for each nurse-transport system pair
TCk = [5]  # Travel cost for the transportation system
PEN = 1.5  # Penalty factor

# Define sets
model.P = RangeSet(1, M)  # Patients
model.N = RangeSet(1, N)  # Nurses
model.K = RangeSet(1, K)  # Transportation systems

# Define decision variables
model.X = Var(model.K, model.P, model.P, model.N, domain=Binary)
model.S = Var(model.P, model.N, domain=NonNegativeReals)
model.O = Var(model.N, model.K, domain=NonNegativeReals)

# Objective function
def objective_rule(model):
    routing_cost = sum(Dij[i-1][j-1] * TCk[k-1] * model.X[k, i, j, n]
                       for k in model.K for i in model.P for j in model.P for n in model.N)
    penalty_cost = sum(model.O[n, k] * TCk[k-1] * PEN
                       for n in model.N for k in model.K)
    return routing_cost + penalty_cost
model.Obj = Objective(rule=objective_rule, sense=minimize)

# Constraints

# (2) Each patient is visited exactly once
def visit_once_rule(model, i):
    return sum(model.X[k, i, j, n] for k in model.K for j in model.P for n in model.N) == 1
model.VisitOnce = Constraint(model.P, rule=visit_once_rule)

# (3) Route assignment
def assignment_rule(model, i, j, n):
    return sum(model.X[k, i, j, n] for k in model.K) == 1
model.Assignment = Constraint(model.P, model.P, model.N, rule=assignment_rule)

# (4) Vehicle capacity (drugs)
def capacity_rule_drugs(model, k, n):
    return sum(Ai[i-1] * model.X[k, i, j, n] for i in model.P for j in model.P) <= CAPk[k-1]
model.CapacityDrugs = Constraint(model.K, model.N, rule=capacity_rule_drugs)

# (5) Vehicle capacity (biological samples)
def capacity_rule_samples(model, k, n):
    return sum(Bi[i-1] * model.X[k, i, j, n] for i in model.P for j in model.P) <= CAPk[k-1]
model.CapacitySamples = Constraint(model.K, model.N, rule=capacity_rule_samples)

# (6) Start of each nurse route
def start_end_rule(model, n):
    return sum(model.X[k, 1, j, n] for k in model.K for j in model.P) == 1
model.StartEnd = Constraint(model.N, rule=start_end_rule)

# (7) Flow conservation
def flow_conservation_rule(model, h, k, n):
    return (sum(model.X[k, i, h, n] for i in model.P) ==
            sum(model.X[k, h, j, n] for j in model.P))
model.FlowConservation = Constraint(model.P, model.K, model.N, rule=flow_conservation_rule)

# (8) Ensure the route starts
def ensure_start_rule(model, k, n):
    return sum(model.X[k, i, 1, n] for i in model.P) == 1
model.EnsureStart = Constraint(model.K, model.N, rule=ensure_start_rule)

# (9) Time sequencing
def time_sequence_rule(model, i, j, n):
    BIG = 1000
    return (model.S[i, n] + Dij[i-1][j-1] + Wi[i-1] -
            BIG * (1 - sum(model.X[k, i, j, n] for k in model.K)) <= model.S[j, n])
model.TimeSequence = Constraint(model.P, model.P, model.N, rule=time_sequence_rule)

# (10) Time window (service start)
def time_window_rule_start(model, i, n):
    return model.S[i, n] >= Ei[i-1]
model.TimeWindowStart = Constraint(model.P, model.N, rule=time_window_rule_start)

# (11) Time window (service end)
def time_window_rule_end(model, i, n):
    return model.S[i, n] <= Li[i-1]
model.TimeWindowEnd = Constraint(model.P, model.N, rule=time_window_rule_end)

# (12) Distance traveled
def distance_calculation_rule(model, n, k):
    return (model.O[n, k] == sum(Dij[i-1][j-1] * model.X[k, i, j, n]
                                 for i in model.P for j in model.P))
model.DistanceCalculation = Constraint(model.N, model.K, rule=distance_calculation_rule)

# Solve the model using the GLPK solver
solver = SolverFactory('glpk')
results = solver.solve(model, tee=True)

# Check solver status
if (results.solver.status == SolverStatus.ok and
    results.solver.termination_condition == TerminationCondition.optimal):
    print("\nOptimal Solution Found!")
    print(f"Objective Value: {value(model.Obj)}")

else:
    print("\nNo optimal solution found.")
    print(f"Solver Status: {results.solver.status}")
    print(f"Termination Condition: {results.solver.termination_condition}")


GLPSOL--GLPK LP/MIP Solver 5.0
Parameter(s) specified in the command line:
 --write /tmp/tmpeha7hufi.glpk.raw --wglp /tmp/tmpefkzhb2x.glpk.glp --cpxlp
 /tmp/tmpua7wubrb.pyomo.lp
Reading problem data from '/tmp/tmpua7wubrb.pyomo.lp'...
/tmp/tmpua7wubrb.pyomo.lp:427: warning: lower bound of variable 'x17' redefined
/tmp/tmpua7wubrb.pyomo.lp:427: warning: upper bound of variable 'x17' redefined
67 rows, 26 columns, 176 non-zeros
18 integer variables, all of which are binary
445 lines were read
Writing problem data to '/tmp/tmpefkzhb2x.glpk.glp'...
353 lines were written
GLPK Integer Optimizer 5.0
67 rows, 26 columns, 176 non-zeros
18 integer variables, all of which are binary
Preprocessing...
PROBLEM HAS NO PRIMAL FEASIBLE SOLUTION
Time used:   0.0 secs
Memory used: 0.1 Mb (60964 bytes)
Writing MIP solution to '/tmp/tmpeha7hufi.glpk.raw'...
102 lines were written

No optimal solution found.
Solver Status: ok
Termination Condition: infeasible
